# Groundwater corrosivity in Telangana: indices + ML screening

**Question:** is groundwater aggressive to steel/GI pipes, and can we guess that from cheap field measurements (pH, EC) instead of a full lab analysis?

**Steps:** download data -> clean + QC -> compute corrosivity indices -> label risk -> explore -> ML (tested on unseen districts and a unseen year) -> a small "check your water" function.

Run all cells top to bottom (about 5-8 minutes on Colab). Runtime > Run all.

**Honest scope:** one state (Telangana, 2018-2020, pre/post-monsoon). Labels are computed from water chemistry, not measured pipe corrosion.

## 1. Setup

In [ ]:
!pip -q install shap openpyxl
import re, io, glob, zipfile, warnings
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
SEED = 42
RISK_ORDER = ["Scaling", "Balanced", "Moderate", "High"]
RISK_COLORS = {"Scaling": "#2a78d6", "Balanced": "#898781", "Moderate": "#eb6834", "High": "#a8281f"}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e1e0d9", "axes.axisbelow": True, "figure.dpi": 110})

## 2. Download the data
Two public Kaggle redistributions of the Telangana State groundwater survey (post-monsoon 2018-2020, pre-monsoon 2018-2020).
Original source and licence are **not verified**; check before reusing publicly.
If the download fails, download both datasets manually from Kaggle and upload the CSV/XLSX files into `data/post` and `data/pre`.

In [ ]:
KAGGLE = {"post": "sivapriyagarladinne/telangana-post-monsoon-ground-water-quality-data",
          "pre": "sadiyazubair/telangana-ground-water-classification"}

def download(slug, folder):
    r = requests.get(f"https://www.kaggle.com/api/v1/datasets/download/{slug}", timeout=180)
    r.raise_for_status()
    zipfile.ZipFile(io.BytesIO(r.content)).extractall(folder)

for key, slug in KAGGLE.items():
    download(slug, f"data/{key}")
print(sorted(glob.glob("data/*/*")))

## 3. Load and harmonise
The files use different column spellings (`E.C` / `EC`, `CO_-2`, `Ca+2`, ...), so headers are normalised first.
Values like `BDL`, `<0.1` and the typo `8..05` are parsed. Blank carbonate (2019 file) is treated as 0.

In [ ]:
HEADER = [(r"^e\.?c\.?$", "EC"), (r"^t\.?d\.?s$", "TDS"), (r"^t\.?h\.?$", "TH"),
          (r"^co_?3?[\s_+-]*\d*$", "CO3"), (r"^hco[\w\s_+-]*$", "HCO3"), (r"^cl[\s+-]*$", "Cl"),
          (r"^f[\s+-]*$", "F"), (r"^no3[\s+-]*$", "NO3"), (r"^so4[\s+\d-]*$", "SO4"),
          (r"^na[\s+\d]*$", "Na"), (r"^k[\s+]*$", "K"), (r"^ca[\s+\d]*$", "Ca"), (r"^mg[\s+\d]*$", "Mg")]
RENAME = {"district": "District", "mandal": "Mandal", "village": "Village",
          "lat_gis": "Latitude", "long_gis": "Longitude", "gwl": "GWL_m"}
ION_COLS = ["pH", "EC", "TDS", "CO3", "HCO3", "Cl", "F", "NO3", "SO4", "Na", "K", "Ca", "Mg", "TH"]

def canonical(col):
    c = str(col).strip(); low = c.lower()
    for pat, name in HEADER:
        if re.match(pat, low):
            return name
    return RENAME.get(c, c)

def parse_value(v, bdl_fraction=0.5):
    if v is None or (isinstance(v, float) and np.isnan(v)): return np.nan
    if isinstance(v, (int, float, np.number)): return float(v)
    s = str(v).strip()
    m = re.match(r"^\s*<\s*([\d.]+)\s*$", s)
    if m: return bdl_fraction * float(m.group(1))
    if re.match(r"^\s*(bdl|nd|nil)\s*$", s, re.I): return 0.0
    try: return float(re.sub(r"\.{2,}", ".", s.replace(",", "")))
    except ValueError: return np.nan

def parse_season(label):
    s = str(label).lower()
    return ("Post" if "post" in s else "Pre"), int(re.search(r"(20\d\d)", s).group(1))

frames = [pd.read_csv(f) for f in sorted(glob.glob("data/*/*.csv"))]
frames += [pd.read_excel(f) for f in sorted(glob.glob("data/*/*.xlsx"))]
df = pd.concat([f.rename(columns=canonical) for f in frames], ignore_index=True)
df[["Season", "Year"]] = df["season"].apply(lambda v: pd.Series(parse_season(v)))
for c in ION_COLS:
    df[c] = df[c].map(parse_value)
df["District"] = df["District"].astype(str).str.strip().str.upper()
df = df[["District", "Mandal", "Village", "Latitude", "Longitude", "GWL_m", "Year", "Season"] + ION_COLS]
df["CO3"] = df["CO3"].fillna(0.0)
print(df.shape, "|", df.District.nunique(), "districts")
print(df.groupby(["Year", "Season"]).size().unstack())
df.head(3)

## 4. Corrosivity indices
All standard water-chemistry formulas (concentrations in mg/L unless noted):

* **Langelier Saturation Index (LSI)** = pH - pHs. Below 0: water dissolves CaCO3 (corrosive). Above 0: it deposits scale.
* **Ryznar Stability Index (RSI)** = 2 pHs - pH (cross-check only).
* **Larson-Skold index** = (Cl + SO4) / (HCO3 + CO3) in meq/L. Above 1.2: chloride/sulphate strongly attack steel.

**Risk label** (priority order): `High` if LSI < -2 or (LSI < -0.5 and LS > 1.2); `Moderate` if LSI < -0.5 or LS > 1.2; `Scaling` if LSI > 0.5 and LS < 0.8; otherwise `Balanced`.

In [ ]:
EQ = {"Ca": 20.04, "Mg": 12.15, "Na": 22.99, "K": 39.10, "HCO3": 61.02, "CO3": 30.0,
      "Cl": 35.45, "SO4": 48.03, "NO3": 62.0, "F": 19.0}
CA_CACO3, HCO3_CACO3, CO3_CACO3 = 50.04 / 20.04, 50.04 / 61.02, 50.04 / 30.0
EC_TO_TDS = 0.64   # the source files' TDS is exactly 0.64 x EC

def meq(x, ion): return np.asarray(x, dtype=float) / EQ[ion]
def alkalinity(hco3, co3=0.0): return np.asarray(hco3, float) * HCO3_CACO3 + np.asarray(co3, float) * CO3_CACO3

def ph_sat(tds, temp_c, ca, hco3, co3=0.0):
    a = (np.log10(np.asarray(tds, float)) - 1) / 10
    b = -13.12 * np.log10(np.asarray(temp_c, float) + 273) + 34.55
    c = np.log10(np.asarray(ca, float) * CA_CACO3) - 0.4
    d = np.log10(alkalinity(hco3, co3))
    return (9.3 + a + b) - (c + d)

def lsi(ph, tds, t, ca, hco3, co3=0.0): return np.asarray(ph, float) - ph_sat(tds, t, ca, hco3, co3)
def rsi(ph, tds, t, ca, hco3, co3=0.0): return 2 * ph_sat(tds, t, ca, hco3, co3) - np.asarray(ph, float)
def larson_skold(cl, so4, hco3, co3=0.0):
    return (meq(cl, "Cl") + meq(so4, "SO4")) / (meq(hco3, "HCO3") + meq(co3, "CO3"))

def charge_balance_error(d):
    cat = sum(meq(np.nan_to_num(d[i]), i) for i in ("Ca", "Mg", "Na", "K"))
    an = sum(meq(np.nan_to_num(d[i]), i) for i in ("HCO3", "CO3", "Cl", "SO4", "NO3", "F"))
    return (cat - an) / (cat + an) * 100

def risk_class(l, s):
    l, s = np.asarray(l, float), np.asarray(s, float)
    out = np.full(l.shape, "Balanced", dtype=object)
    out[(l > 0.5) & (s < 0.8)] = "Scaling"
    out[(l < -0.5) | (s > 1.2)] = "Moderate"
    out[(l < -2) | ((l < -0.5) & (s > 1.2))] = "High"
    out[np.isnan(l) | np.isnan(s)] = np.nan
    return out if out.ndim else out.item()

# sanity checks against hand-worked values
assert abs(float(lsi(7.5, 320, 25, 150 / CA_CACO3, 120 / HCO3_CACO3)) - (-0.183)) < 0.005
assert abs(float(larson_skold(35.45, 48.03, 61.02)) - 2.0) < 1e-9
assert risk_class(-2.5, 0.1) == "High" and risk_class(1.0, 0.5) == "Scaling" and risk_class(0.0, 0.5) == "Balanced"
print("index checks passed")

## 5. Water temperature (NASA POWER) and quality control
The Langelier index needs water temperature. We use mean annual air temperature from the NASA POWER climatology
(0.5 degree cells) + 1 C as a proxy for groundwater temperature, and test how much that choice matters later.

QC rules: drop rows missing an input, pH outside 4-11, coordinates outside India, negative concentrations, and
charge-balance error above 10% (a standard check that the lab analysis is internally consistent).

In [ ]:
def fetch_t2m(lat, lon, tries=3):
    for i in range(tries):
        try:
            r = requests.get("https://power.larc.nasa.gov/api/temporal/climatology/point",
                             params=dict(parameters="T2M", community="AG", latitude=lat, longitude=lon, format="JSON"), timeout=60)
            r.raise_for_status()
            return float(r.json()["properties"]["parameter"]["T2M"]["ANN"])
        except Exception:
            if i == tries - 1: raise

snap = lambda v: np.floor(np.asarray(v, float) / 0.5) * 0.5 + 0.25
df["cell_lat"], df["cell_lon"] = snap(df.Latitude), snap(df.Longitude)
cells = df[["cell_lat", "cell_lon"]].drop_duplicates().reset_index(drop=True)
cells["T_air_C"] = [fetch_t2m(a, b) for a, b in zip(cells.cell_lat, cells.cell_lon)]
df = df.merge(cells, on=["cell_lat", "cell_lon"], how="left").drop(columns=["cell_lat", "cell_lon"])
print(len(cells), "NASA POWER cells fetched; air temp range", df.T_air_C.min().round(1), "-", df.T_air_C.max().round(1), "C")

In [ ]:
REQUIRED = ["pH", "EC", "Ca", "HCO3", "Cl", "SO4", "Mg", "Na", "K", "Latitude", "Longitude", "T_air_C"]
df["CBE"] = charge_balance_error(df)
n0 = len(df)
have = df.dropna(subset=REQUIRED)
steps = [("pH outside 4-11", have.pH.between(4, 11)),
         ("coordinates outside India", have.Latitude.between(6, 37.5) & have.Longitude.between(68, 97.5)),
         ("negative concentration", ~(have[["Ca", "Mg", "Na", "K", "HCO3", "CO3", "Cl", "SO4"]] < 0).any(axis=1)),
         ("|charge balance error| > 10%", have.CBE.abs() <= 10)]
keep = pd.Series(True, index=have.index); report = [("missing a required input", n0 - len(have))]
for name, ok in steps:
    report.append((name, int((keep & ~ok).sum()))); keep &= ok
data = have[keep].copy().reset_index(drop=True)
print(pd.DataFrame(report, columns=["rule", "rows removed"]).to_string(index=False))
print(f"\n{n0} -> {len(data)} samples")

def add_indices(d, temp_offset=1.0):
    d = d.copy()
    tds = d["TDS"].fillna(d["EC"] * EC_TO_TDS); t = d["T_air_C"] + temp_offset
    a = (tds, t, d.Ca, d.HCO3, d.CO3)
    d["LSI"], d["RSI"] = lsi(d.pH, *a), rsi(d.pH, *a)
    d["LS"] = larson_skold(d.Cl, d.SO4, d.HCO3, d.CO3)
    d["Risk"] = risk_class(d.LSI.to_numpy(), d.LS.to_numpy())
    return d

data = add_indices(data)
data["Risk"] = pd.Categorical(data["Risk"], RISK_ORDER)
data[["pH", "EC", "LSI", "RSI", "LS"]].describe().round(2)

## 6. What does the water look like?

In [ ]:
mix = data.Risk.value_counts().reindex(RISK_ORDER)
print((pd.DataFrame({"n": mix, "%": (100 * mix / mix.sum()).round(1)})).to_string())
data["AtRisk"] = data.Risk.isin(["Moderate", "High"])
data["Driver"] = np.select([(data.LSI < -0.5) & (data.LS > 1.2), data.LSI < -0.5, data.LS > 1.2],
                           ["both", "LSI only", "LS only"], "none")
drv = data[data.AtRisk].Driver.value_counts()
print("\nWhat makes a sample 'at-risk' (Moderate/High):"); print((100 * drv / drv.sum()).round(1).to_string())
print("\nScale-forming (LSI>0.5) AND chloride/sulphate-aggressive (LS>1.2):", int(((data.LSI > .5) & (data.LS > 1.2)).sum()), "samples")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
for k in RISK_ORDER:
    m = data[data.Risk == k]
    ax.scatter(m.LSI, m.LS, s=10, c=RISK_COLORS[k], edgecolor="white", linewidth=.3, label=f"{k} ({len(m)})")
ax.set_yscale("log")
for x in (-2, -.5, .5): ax.axvline(x, c="#898781", lw=.8, ls="--")
for y in (.8, 1.2): ax.axhline(y, c="#898781", lw=.8, ls="--")
ax.set_xlabel("Langelier Saturation Index"); ax.set_ylabel("Larson-Skold (log)")
ax.set_title("Where the two indices agree and disagree", loc="left", fontweight="bold"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
tab = pd.crosstab(data.District, data.Risk, normalize="index").reindex(columns=RISK_ORDER) * 100
tab = tab.loc[(tab.Moderate + tab.High).sort_values().index]
fig, ax = plt.subplots(figsize=(8, 8)); left = np.zeros(len(tab))
for k in RISK_ORDER:
    ax.barh(tab.index, tab[k], left=left, color=RISK_COLORS[k], edgecolor="white", linewidth=1.2, label=k, height=.78); left += tab[k].to_numpy()
ax.set_xlim(0, 100); ax.set_xlabel("% of samples"); ax.grid(axis="y", visible=False)
ax.set_title("Corrosion risk class by district", loc="left", fontweight="bold")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(.5, -.07)); plt.tight_layout(); plt.show()
n_by = data.groupby("District").size()
print("Caution: districts with few samples give noisy percentages:", n_by[n_by < 30].to_dict())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for ax, season in zip(axes, ["Pre", "Post"]):
    s = data[data.Season == season]
    for k in RISK_ORDER:
        m = s[s.Risk == k]; ax.scatter(m.Longitude, m.Latitude, s=12, c=RISK_COLORS[k], edgecolor="white", linewidth=.4, label=f"{k} ({len(m)})")
    ax.set_title(f"{season}-monsoon wells", loc="left", fontweight="bold"); ax.set_xlabel("Longitude"); ax.set_aspect("equal")
    ax.legend(loc="upper center", bbox_to_anchor=(.5, -.14), ncol=2, fontsize=8)
axes[0].set_ylabel("Latitude"); plt.tight_layout(); plt.show()

In [ ]:
season = data.groupby(["Year", "Season"]).agg(n=("Risk", "size"), pct_at_risk=("AtRisk", lambda s: round(100 * s.mean(), 1)),
                                              median_LSI=("LSI", lambda s: round(s.median(), 2)))
print(season.to_string())
w = data.pivot_table(index=["Latitude", "Longitude", "Year"], columns="Season", values="LSI").dropna()
print(f"\nSame well, same year ({len(w)} pairs): median LSI pre {w.Pre.median():.2f} -> post {w.Post.median():.2f};"
      f" post-monsoon LSI is higher at {100 * (w.Post > w.Pre).mean():.0f}% of wells")

## 7. How much does the temperature assumption matter?
We assumed water is air temperature + 1 C. Recompute the labels under other assumptions and count how many change.

In [ ]:
base = data.Risk.astype(str)
rows = []
for name, tair, off in [("air T + 0 C", None, 0), ("air T + 1 C (baseline)", None, 1), ("air T + 3 C", None, 3),
                        ("fixed 20 C", 19, 1), ("fixed 30 C", 29, 1)]:
    x = data.copy()
    if tair is not None: x["T_air_C"] = tair
    r = add_indices(x, off).Risk.astype(str)
    rows.append((name, int((r != base).sum()), round(100 * (r != base).mean(), 1)))
print(pd.DataFrame(rows, columns=["scenario", "labels changed", "%"]).to_string(index=False))

## 8. Machine learning: can cheap measurements predict the risk class?
**Target (3 classes):** `At-risk` (Moderate + High; High alone has too few samples), `Balanced`, `Scaling`.

**No leakage:** the model never sees Ca, HCO3, CO3, Cl, SO4 or the indices (the label is computed from them).

| Feature set | Inputs |
|---|---|
| A0 | pH, EC (what a cheap field meter gives) |
| A | pH, EC, latitude, longitude, season, air temperature |
| B | location, season, air temperature only (no water measurement) |

**Honest validation:** wells in the same district resemble each other, so a random split looks better than reality.
We test on **unseen districts** (GroupKFold by district) and on **an unseen year** (train 2018-19, test 2020).

In [ ]:
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, f1_score, recall_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data["y"] = data.Risk.astype(str).replace({"Moderate": "At-risk", "High": "At-risk"})
data["SeasonPre"] = (data.Season == "Pre").astype(int)
FEATURES = {"A0 pH+EC": ["pH", "EC"],
            "A  pH+EC+location+season+climate": ["pH", "EC", "Latitude", "Longitude", "SeasonPre", "T_air_C"],
            "B  location+season+climate only": ["Latitude", "Longitude", "SeasonPre", "T_air_C"]}

def make_models():
    return {"Majority class": DummyClassifier(strategy="most_frequent"),
            "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")),
            "Random forest": RandomForestClassifier(n_estimators=150, min_samples_leaf=3, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED),
            "Hist. gradient boosting": HistGradientBoostingClassifier(class_weight="balanced", max_depth=4, learning_rate=.05, max_iter=150, random_state=SEED)}

SCHEMES = {"unseen districts": list(GroupKFold(5).split(data, data.y, data.District)),
           "train 2018-19, test 2020": [(np.where(data.Year < 2020)[0], np.where(data.Year == 2020)[0])]}

def evaluate(model, X, y, folds):
    pred = pd.Series(index=y.index, dtype=object)
    for tr, te in folds:
        m = clone(model).fit(X.iloc[tr], y.iloc[tr]); pred.iloc[te] = m.predict(X.iloc[te])
    idx = pred.dropna().index; yt, yp = y.loc[idx], pred.loc[idx]
    return dict(macro_F1=f1_score(yt, yp, average="macro"), accuracy=(yt == yp).mean(),
                recall_at_risk=recall_score(yt, yp, labels=["At-risk"], average=None)[0]), yp

rows, oof = [], {}
for fname, cols in FEATURES.items():
    for sname, folds in SCHEMES.items():
        for mname, model in make_models().items():
            if fname.startswith("B") and mname == "Logistic regression": continue
            res, yp = evaluate(model, data[cols], data.y, folds)
            rows.append(dict(features=fname, validation=sname, model=mname, **res)); oof[(fname, sname, mname)] = yp
results = pd.DataFrame(rows)
results.round(3)

In [ ]:
best = results[results.model != "Majority class"].sort_values("macro_F1", ascending=False).groupby(["features", "validation"]).head(1)
print("Best model per setting (majority-class guess scores macro-F1 ~0.20; chance for 3 classes ~0.33):\n")
print(best.sort_values(["features", "validation"]).round(3).to_string(index=False))

In [ ]:
key = ("A  pH+EC+location+season+climate", "unseen districts", "Random forest")
yp = oof[key]; classes = ["Scaling", "Balanced", "At-risk"]
cm = confusion_matrix(data.y.loc[yp.index], yp, labels=classes); pct = cm / cm.sum(1, keepdims=True) * 100
fig, ax = plt.subplots(figsize=(5, 4.2)); ax.imshow(pct, cmap="Blues", vmin=0, vmax=100); ax.grid(False)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{pct[i, j]:.0f}%\n(n={cm[i, j]})", ha="center", va="center", color="white" if pct[i, j] > 55 else "black", fontsize=9)
ax.set_xticks(range(3), classes); ax.set_yticks(range(3), classes); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual (row %)")
ax.set_title("Random forest on unseen districts", loc="left", fontweight="bold", fontsize=10); plt.tight_layout(); plt.show()

### What drives the predictions? (SHAP)

In [ ]:
import shap
cols = FEATURES["A  pH+EC+location+season+climate"]
rf = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED).fit(data[cols], data.y)
sample = data[cols].sample(600, random_state=SEED)
sv = shap.TreeExplainer(rf).shap_values(sample)
k = list(rf.classes_).index("At-risk")
vals = sv[:, :, k] if getattr(sv, "ndim", 0) == 3 else sv[k]
shap.summary_plot(vals, sample, show=False, plot_size=(7.5, 4)); plt.title("Pushes toward 'At-risk'", loc="left", fontweight="bold"); plt.tight_layout(); plt.show()
print(pd.Series(np.abs(vals).mean(0), index=cols).sort_values(ascending=False).round(3).to_string())

## 9. Check your water
Two modes: exact indices from a full lab report, or an ML screening estimate from pH and EC only.

In [ ]:
ADVICE = {"High": "Avoid GI/mild steel. Use uPVC, HDPE or CPVC, or lined pipe. Inspect hand pumps more often.",
          "Moderate": "Chloride/sulphate attack on steel is likely. Prefer plastic pipe; if GI is used, plan a shorter replacement cycle.",
          "Balanced": "Any standard pipe material is fine.",
          "Scaling": "Expect carbonate scale and clogging. Plan descaling; avoid very small bore pipe.",
          "At-risk": "Water looks aggressive toward steel. Prefer uPVC/HDPE; confirm with a lab test."}

def check_lab(ph, ec, temp_c, ca, hco3, cl, so4, co3=0.0):
    tds = ec * EC_TO_TDS
    L, R, S = float(lsi(ph, tds, temp_c, ca, hco3, co3)), float(rsi(ph, tds, temp_c, ca, hco3, co3)), float(larson_skold(cl, so4, hco3, co3))
    risk = risk_class(L, S)
    print(f"LSI {L:.2f} | RSI {R:.2f} | Larson-Skold {S:.2f}  ->  {risk}\n{ADVICE[risk]}")

screen_A0 = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight="balanced_subsample", random_state=SEED).fit(data[["pH", "EC"]], data.y)
def screen(ph, ec):
    p = screen_A0.predict_proba(pd.DataFrame({"pH": [ph], "EC": [ec]}))[0]
    for c, v in sorted(zip(screen_A0.classes_, p), key=lambda t: -t[1]): print(f"  {c:9s} {v:.0%}")
    print(ADVICE[screen_A0.classes_[p.argmax()]], "\n(Screening estimate only; confirm with a lab analysis.)")

print("Hard, alkaline water:"); check_lab(ph=8.4, ec=900, temp_c=28, ca=120, hco3=400, cl=30, so4=10)
print("\nSoft, chloride-rich water:"); check_lab(ph=6.5, ec=1500, temp_c=28, ca=10, hco3=30, cl=400, so4=40)
print("\nQuick screen, pH 7.8, EC 2000:"); screen(7.8, 2000)

## 10. What we found (from the original run) and limitations

* About **43%** of samples are Moderate/High risk, but **~95%** of those are flagged by Larson-Skold alone. Telangana water is mostly alkaline and hard (scale-forming), so 'Moderate' probably **overstates** real corrosion risk where the two indices disagree.
* Dry-season (pre-monsoon) water is more often at risk than post-monsoon.
* Labels are robust to the temperature assumption (about 1-2% change for plausible values).
* **Screening from pH + EC is only moderately accurate** (macro-F1 about 0.58-0.63 on unseen districts/year, majority guess 0.20; about 30% of truly at-risk samples are missed). EC (salinity) matters most. Use it for triage, not for deciding pipe material.
* **Location alone does not predict risk** (near chance on unseen districts).

**Limitations:** one state; labels are computed from chemistry, not measured pipe failures; air temperature stands in for water temperature; data licence/provenance unverified; no iron column, so no independent cross-check.

**Possible next steps:** simplify to a binary "aggressive vs not" label; add more states when national data is accessible; validate against real hand-pump failure records.